# Melodyne Backprop Lab · Restore the synth

You can hear how a short synth phrase should sound. You also have the same notes played through a damaged preset: thin bass, heavy mids, muted treble, and too much drive. Which way should each control move?

In [SmartVal](../01-smartval-neural-networks-and-backprop.ipynb), you traced error back through weighted sums and ReLU gates. Here the same chain rule travels through frequency processing and a smooth distortion curve. The model is an audio renderer, not a neural network; only **four raw control values** are trainable.

Melodyne Labs is the scenario name. This is a self-contained synthetic restoration experiment, not a plugin integration or a claim to recover an arbitrary recording's instrument.

## 0 · Keep the notes. Fix how they sound.

| The question | What the experiment keeps visible |
| --- | --- |
| What stays fixed? | One eight-note phrase, its timing, and the renderer |
| What can change? | Bass, mid, treble, and drive, controlled by four raw variables |
| Where does error come from? | Candidate audio compared with reference audio |
| Can the gradient be trusted? | All four slopes checked with two-sided nudges |
| Did optimization help? | Actual listening checkpoints, loss history, and separate recovery checks |

The code generates a 1.12-second phrase locally. No recordings or pretrained weights are loaded. The known target preset lets us check recovery, but its knob values are not supplied to the loss.

Use the parent chapter's TensorFlow-capable kernel. The setup cell imports NumPy, TensorFlow, plotting, and audio playback, and sets seeds. The optimizer loop is explicitly placed on CPU. If the environment is not ready, use the [existing setup instructions](README.md#setup); this optional branch does not block the PyTorch route.

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from IPython.display import Audio, display

SEED = 23
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.get_logger().setLevel("ERROR")
print(f"TensorFlow {tf.__version__} | NumPy {np.__version__}")
print("Optimization device: CPU (selected explicitly below)")

## 1 · Give every tone control something to work with

The phrase is `A2, C#3, E3, D3, G3, E3, B2, A2`. Each note lasts `0.14 s`. At `8,000 samples/s`, that is `1,120` amplitude values per note and `8,960` for the whole phrase.

A waveform is a list of air-pressure-like amplitudes over time. A pure sine at 110 Hz makes 110 cycles per second. The code adds eighteen **harmonics**, sine waves at integer multiples of each note's base frequency, with amplitudes decreasing as `1/harmonic`. The high components give the treble control something to affect; bass-only input would make a treble setting hard to infer.

Read the cell in four pieces:

1. `synth_note` creates a time vector and adds the harmonic waves. Note index changes their phase, the point in the cycle where each starts.
2. A cosine fade ramps the first and last `144` samples of each note. This smooths note boundaries instead of switching from silence to a large amplitude in one sample.
3. `concatenate` joins eight `(1120,)` notes **end to end** into one `(8960,)` timeline. Stacking would make a different, note-by-sample table.
4. The final scaling sets the largest absolute dry amplitude to `0.45`. `dry` becomes a fixed TensorFlow tensor, and the audio player lets you hear it.

This is the **dry signal**, before the adjustable effects. The prose need not guess what it sounds like: listen to the actual array. `normalize=False` keeps the player from independently turning every comparison clip up to the same peak.

In [ ]:
SAMPLE_RATE = 8_000
NOTE_SECONDS = 0.14
NOTE_SAMPLES = int(SAMPLE_RATE * NOTE_SECONDS)
PHRASE_HZ = np.array([110.00, 138.59, 164.81, 146.83, 196.00, 164.81, 123.47, 110.00], dtype=np.float32)

def synth_note(frequency_hz, note_index):
    time = np.arange(NOTE_SAMPLES, dtype=np.float32) / SAMPLE_RATE
    tone = np.zeros_like(time)
    for harmonic in range(1, 19):
        harmonic_phase = 0.17 * note_index + 0.11 * harmonic
        tone += np.sin(2.0 * np.pi * harmonic * frequency_hz * time + harmonic_phase) / harmonic
    fade_samples = int(0.018 * SAMPLE_RATE)
    fade = 0.5 - 0.5 * np.cos(np.linspace(0.0, np.pi, fade_samples, dtype=np.float32))
    envelope = np.ones(NOTE_SAMPLES, dtype=np.float32)
    # Fill the first/last (144,) slices of the (1120,) envelope; reverse the fade
    # at the end so both note edges taper smoothly without changing the note length.
    envelope[:fade_samples] = fade
    envelope[-fade_samples:] = fade[::-1]
    return tone * envelope

# Eight (1120,) notes -> one (8960,) timeline: concatenate end-to-end, not as parallel channels.
dry_np = np.concatenate([synth_note(hz, index) for index, hz in enumerate(PHRASE_HZ)])
dry_np = (0.45 * dry_np / np.max(np.abs(dry_np))).astype(np.float32)
dry = tf.constant(dry_np)
time_axis = np.arange(dry_np.size) / SAMPLE_RATE
print(f"Phrase: {dry_np.size / SAMPLE_RATE:.2f} seconds, {dry_np.size:,} samples at {SAMPLE_RATE:,} Hz")
print("Harmonics span the bass, mid, and treble masks so every tone control is observable.")
display(Audio(dry_np, rate=SAMPLE_RATE, normalize=False))
assert dry_np.size == len(PHRASE_HZ) * NOTE_SAMPLES
assert np.max(np.abs(dry_np)) <= 0.451

### Same performance, different tone

Now keep those 8,960 sample positions and the note sequence fixed. We'll change frequency balance with bass, mid, and treble, then change the waveform's shape with drive.

A **differentiable** operation provides a local slope, so a small change in a control can be followed through to a small change in error. That is the useful property here. It does not mean that every setting is uniquely recoverable or that every gradient is large.

Two representations will share the work:

- **Time domain:** amplitude at each sample, useful for comparing aligned waveform shape.
- **Frequency domain:** complex FFT coefficients, useful for changing the strength of low and high frequency components.

The **FFT** is a change of representation, not a new source of sound. Its inverse brings the adjusted frequency content back to the same timeline.

![The fixed eight-note phrase passes through adjustable tone and drive controls. Audio error traces responsibility back to four raw controls before a separate optimizer changes them.](images/01-melodyne-fixed-notes-adjustable-tone.png)

The reference and candidate share notes and timing. Only the candidate's four raw controls will receive optimizer updates; the target audio stays fixed.

## 2 · Build a renderer that a gradient can pass through

First solve a practical problem: an optimizer can request any real number, but a gain should stay in its allowed range. Keep an unrestricted **raw** value for each control, then pass it through a sigmoid and stretch it between that control's limits.

For illustration, raw value `0` becomes sigmoid value `0.5`, the midpoint. Bass, mid, and treble each span `0.35 to 1.65`, so their midpoint is `1.00`. Drive spans `0.60 to 2.40`, so its midpoint is `1.50`. These are gains and a drive amount, not decibels or notes.

`decode_controls` performs that smooth conversion. `encode_controls` reverses it for the strictly interior starting and target settings. The allowed endpoints would require infinite raw values. Far out in raw space, sigmoid also flattens and its derivative becomes small: bounded does not mean equally easy to optimize everywhere.

Next, three overlapping **frequency masks** assign each FFT bin a mixture of bass, mid, and treble influence. The fixed centres are 120, 520, and 2,200 Hz; the masks are smooth, not three hard frequency cutoffs. The code compares distances on a logarithmic frequency scale, where ratios matter more than equal gaps in Hz. It turns three closeness scores into positive shares with **softmax**: exponentiate each score, then divide by their sum. Applying this along the three-band axis makes the weights sum to one at every bin. This is fixed blending, not a learned probability that a note belongs to a band.

An illustrative bin with mask weights `[0.8, 0.2, 0]` and gains `[1.30, 0.72, 1.18]` receives gain `0.8*1.30 + 0.2*0.72 = 1.184`. These mask weights illustrate blending; they are not a measured mask row from the notebook.

> **Visual to add — Overlapping tone stencils**
>
> Draw a harmonic spectrum for the fixed phrase with translucent bass, mid, and treble stencils overlapping it. Beside one illustrative bin show mask shares 0.8, 0.2, 0 and the blended gain 1.184. Keep the gain multiplication before inverse FFT and drive. Do not draw each frequency belonging exclusively to one knob, or show tone gains changing the note-event list.
>
> **Proposed caption:** “A frequency can listen to more than one tone control.”

Read the long cell as this shape trace:

```text
raw controls (4,) -> bounded gains/drive (4,)
fixed masks (4481,3) @ tone gains (3,) -> response (4481,)
dry samples (8960,) -> rFFT complex bins (4481,)
                   -> multiply one real gain per matching bin
                   -> inverse FFT samples (8960,)
                   -> smooth drive -> candidate audio (8960,)
```

A real-input FFT needs only `N/2 + 1 = 4,481` bins here; the other frequency coefficients are redundant. Complex coefficients carry amplitude and phase. Multiplying each by a real gain changes its size while retaining phase; the inverse FFT reconstructs the adjusted waveform.

Finally, `tanh(drive * equalized) / tanh(drive)` smoothly rounds large amplitudes. Unlike hard clipping, it has a smooth derivative. The denominator fixes the response at an input of 1; it does not keep overall loudness constant or guarantee an arbitrary input stays below amplitude 1. The later assertions check the actual target and damaged clips.

The masks and note generator are fixed. Only the four raw controls are adjusted.

In [ ]:
CONTROL_NAMES = ("bass", "mid", "treble", "drive")
CONTROL_LOW = tf.constant([0.35, 0.35, 0.35, 0.60], dtype=tf.float32)
CONTROL_HIGH = tf.constant([1.65, 1.65, 1.65, 2.40], dtype=tf.float32)

def decode_controls(raw_controls):
    # All vectors are (4,): each raw value uses its own bounds, keeping control identity intact.
    return CONTROL_LOW + (CONTROL_HIGH - CONTROL_LOW) * tf.sigmoid(raw_controls)

def encode_controls(controls):
    position = (tf.convert_to_tensor(controls, tf.float32) - CONTROL_LOW) / (CONTROL_HIGH - CONTROL_LOW)
    return tf.math.log(position / (1.0 - position))

frequency_hz = tf.cast(tf.range(dry_np.size // 2 + 1), tf.float32) * SAMPLE_RATE / dry_np.size
log_frequency = tf.math.log(frequency_hz + 45.0) / tf.math.log(2.0)
center_hz = tf.constant([120.0, 520.0, 2_200.0], dtype=tf.float32)
center_log = tf.math.log(center_hz + 45.0) / tf.math.log(2.0)
band_width = tf.constant([1.05, 0.95, 1.15], dtype=tf.float32)
# (4481,) -> (4481, 1) frequencies; centers/widths (3,) -> (1, 3).
# Broadcasting -> (4481, 3) compares every frequency with every tone band.
band_logits = -0.5 * tf.square((log_frequency[:, None] - center_log[None, :]) / band_width[None, :])
# Keep (4481, 3); normalize the three bands per frequency so their gains blend, not accumulate.
band_masks = tf.nn.softmax(band_logits, axis=1)

def render(raw_controls):
    # (4,) -> four scalars: separate the three EQ gains from the later drive control.
    bass, mid, treble, drive = tf.unstack(decode_controls(raw_controls))
    # Stack three scalars -> (3,); (4481, 3) @ (3,) -> (4481,), one gain per FFT bin.
    response = tf.linalg.matvec(band_masks, tf.stack([bass, mid, treble]))
    # rfft: (8960,) samples -> (4481,) complex bins; multiply matching gains bin-by-bin.
    # irfft returns (8960,); explicit length keeps candidate and target samples aligned.
    equalized = tf.signal.irfft(
        tf.signal.rfft(dry) * tf.cast(response, tf.complex64),
        fft_length=[dry_np.size],
    )
    # Scalar drive broadcasts over (8960,) samples: one shared knob shapes the whole phrase.
    return tf.math.tanh(drive * equalized) / tf.math.tanh(drive)

assert band_masks.shape == (dry_np.size // 2 + 1, 3)
np.testing.assert_allclose(tf.reduce_sum(band_masks, axis=1).numpy(), 1.0, atol=1e-6)

### Hide a preset, then damage it

The next cell renders two versions of the same dry phrase:

- Target controls: `[bass 1.30, mid 0.72, treble 1.18, drive 1.55]`.
- Damaged controls: `[bass 0.45, mid 1.50, treble 0.52, drive 2.25]`.

These are the actual configured values, not learned results. `encode_controls` turns them into raw values; `render` makes the audio. `stop_gradient` fixes the reference as something to compare with, not something that can move to make the candidate look better.

**Predict, then listen:** should training need to change the notes to fix this damage? No: the notes never changed. The difference is entirely in the tone and drive path.

We will keep target knob values out of the objective. Otherwise the optimizer could simply chase the known controls and we would not test whether audio error can guide them.

In [ ]:
target_controls = tf.constant([1.30, 0.72, 1.18, 1.55], dtype=tf.float32)
damaged_controls = tf.constant([0.45, 1.50, 0.52, 2.25], dtype=tf.float32)
target_raw = encode_controls(target_controls)
damaged_raw = encode_controls(damaged_controls)
target_audio = tf.stop_gradient(render(target_raw))
damaged_audio = tf.stop_gradient(render(damaged_raw))

print("Secret target:  " + ", ".join(f"{name}={value:.2f}" for name, value in zip(CONTROL_NAMES, target_controls.numpy())))
print("Damaged start: " + ", ".join(f"{name}={value:.2f}" for name, value in zip(CONTROL_NAMES, damaged_controls.numpy())))
print("Target reference:")
display(Audio(target_audio.numpy(), rate=SAMPLE_RATE, normalize=False))
print("Damaged starting point:")
display(Audio(damaged_audio.numpy(), rate=SAMPLE_RATE, normalize=False))
assert float(tf.reduce_max(tf.abs(target_audio))) < 1.0
assert float(tf.reduce_max(tf.abs(damaged_audio))) < 1.0

## 3 · Hear a mismatch, turn it into a loss

Two aligned clips can differ in sample shape and in frequency balance. We'll score both:

- **Waveform MSE:** subtract corresponding amplitudes, square, and average across 8,960 samples. It notices timing and phase differences too; this works because our clips are exactly aligned.
- **Log-spectrum MSE:** compare `log(1 + 20*magnitude)` across 4,481 FFT bins. The logarithm compresses the range so strong components do not completely hide weaker ones. Magnitudes discard phase, so this term cannot replace the waveform view.

The scalar objective is `waveform_loss + 0.05*spectrum_loss`. The coefficient is a chosen trade-off between two differently scaled scores, not a universal perceptual weighting. `log_magnitude_spectrum` adds a tiny `1e-12` inside the square root to avoid an awkward zero-magnitude slope.

The code compares candidate **audio** with fixed target audio. It never subtracts target knobs from candidate knobs in this objective.

`GradientTape` records the operations that used `probe_raw`, a variable initialized at the damaged preset. `tape.gradient` then returns four slopes in `[bass, mid, treble, drive]` order. If a slope is positive, a sufficiently small decrease in that raw value should lower loss nearby. A negative slope suggests a small increase.

> **Visual to add — Two knob scales, one backward path**
>
> Draw a raw-control slider linked by a sigmoid cam to the bounded bass dial [0.35,1.65], with raw 0 mapping to gain 1.00. Continue through candidate audio into a comparison against a fixed reference. Draw the derivative returning to the raw slider through the cam, and a separate optimizer hand moving it afterward. Do not label the derivative as “per bass gain unit”: the checked quantity is loss per raw unit. Do not let the reference move.
>
> **Proposed caption:** “The gradient includes the linkage between the raw parameter and the audible knob.”

Backpropagation computes those local sensitivities. It has not yet changed any setting. The assertion that target loss is near zero is a useful self-check: rendering the reference controls should reproduce the reference audio.

In [ ]:
def log_magnitude_spectrum(audio):
    # (8960,) real samples -> (4481,) bins, so target and candidate compare the same frequencies.
    spectrum = tf.signal.rfft(audio)
    power = tf.square(tf.math.real(spectrum)) + tf.square(tf.math.imag(spectrum))
    return tf.math.log1p(20.0 * tf.sqrt(power + 1e-12))

target_log_spectrum = log_magnitude_spectrum(target_audio)

def restoration_loss(raw_controls):
    estimate = render(raw_controls)
    # Match (8960,) waveform samples and (4481,) spectrum bins separately;
    # reduce each comparison to a scalar before combining the two kinds of error.
    waveform_loss = tf.reduce_mean(tf.square(estimate - target_audio))
    spectrum_loss = tf.reduce_mean(tf.square(log_magnitude_spectrum(estimate) - target_log_spectrum))
    return waveform_loss + 0.05 * spectrum_loss

probe_raw = tf.Variable(damaged_raw)
with tf.GradientTape() as tape:
    probe_loss = restoration_loss(probe_raw)
autodiff_gradient = tape.gradient(probe_loss, probe_raw)

print(f"Forward-pass loss: {float(probe_loss):.6f}")
print("Backprop responsibility at the damaged preset:")
for name, gradient in zip(CONTROL_NAMES, autodiff_gradient.numpy()):
    direction = "decrease raw control" if gradient > 0 else "increase raw control"
    print(f"  dL/d{name:6s} = {gradient:+.7f} -> {direction}")
assert float(restoration_loss(target_raw)) < 1e-10
assert np.all(np.isfinite(autodiff_gradient.numpy()))

### A plausible direction is not enough

An automatic gradient can look sensible even when a signal path was accidentally detached. Check it independently: nudge one raw value up, render and score; nudge it down, render and score again.

Keep all other controls fixed. We want the slope for one coordinate, not the combined effect of moving several knobs.

## 4 · Audit all four raw-control slopes

The numerical slope is `(loss_plus - loss_minus) / (2*epsilon)`. With `epsilon = 0.001`, the two trial settings are 0.002 raw units apart. Four coordinates require eight extra forward renders.

`tf.one_hot` makes an offset vector with only the selected coordinate nonzero. Stacking the four estimated slopes produces shape `(4,)`, matching `GradientTape`'s result and control order.

The check uses both relative tolerance `0.03` and absolute tolerance `0.00002`. Finite-difference estimates and float32 autodiff need not be bit-identical; making the nudge arbitrarily small would eventually magnify numerical noise rather than improve the check.

This is independent differentiation, **not** an independent renderer: both methods evaluate the same audio computation and loss. Agreement checks the gradient of this local implementation, not whether we chose the right restoration objective.

In [ ]:
epsilon = tf.constant(1e-3, dtype=tf.float32)
finite_difference_gradient = []
for index in range(len(CONTROL_NAMES)):
    # (4,) has one nonzero entry: nudge only this raw control, leaving the other three fixed.
    offset = tf.one_hot(index, len(CONTROL_NAMES), dtype=tf.float32) * epsilon
    loss_plus = restoration_loss(probe_raw + offset)
    loss_minus = restoration_loss(probe_raw - offset)
    finite_difference_gradient.append((loss_plus - loss_minus) / (2.0 * epsilon))
# Four scalar slopes -> (4,), matching GradientTape's raw-control order for a fair check.
finite_difference_gradient = tf.stack(finite_difference_gradient)

print("control   GradientTape    finite diff    absolute error")
for name, automatic, numerical in zip(CONTROL_NAMES, autodiff_gradient.numpy(), finite_difference_gradient.numpy()):
    print(f"{name:7s}  {automatic:+.8f}   {numerical:+.8f}   {abs(automatic - numerical):.2e}")
np.testing.assert_allclose(autodiff_gradient.numpy(), finite_difference_gradient.numpy(), rtol=3e-2, atol=2e-5)
print("All four raw-control gradients pass the central-difference check.")

## 5 · Let the optimizer act, then listen

Now create a fresh trainable `raw_controls` variable at the damaged settings and an **Adam optimizer** with learning rate `0.06`. Adam uses gradients plus running summaries of their size and direction to choose per-coordinate updates. It is not simply “subtract 0.06 times every gradient.”

Each of 220 updates does four jobs:

```text
current raw controls -> candidate audio -> scalar loss
                    -> four gradients -> Adam changes raw controls
                    -> render again when a listening checkpoint is due
```

There is one complete phrase, not a dataset of songs or a mini-batch loader. A “listening batch” here means 25 optimizer updates between audio checkpoints, not 25 training examples.

Read the long training cell around its three responsibilities:

1. `play_training_checkpoint` captures copies of the **current** controls and audio, recomputes current loss, and plays the result. These snapshots are not involved in differentiation.
2. The loop records loss and decoded controls **before** each update, then applies the update. Checkpoints after updates 25, 50, and so on therefore represent completed updates.
3. After update 220, the final state is appended. Histories have 221 states: initial plus one per completed update. Audio checkpoints are `0, 25, 50, 75, 100, 125, 150, 175, 200, 220`.

The target reference is replayed for comparison. Use the printed loss and knob settings beside each player; “it sounds closer” is useful evidence, but your ears are not an exact derivative test.

In [ ]:
raw_controls = tf.Variable(damaged_raw, name="raw_synth_controls")
optimizer = tf.keras.optimizers.Adam(learning_rate=0.06)
STEPS = 220
LISTEN_EVERY = 25
loss_history = []
knob_history = []
audio_checkpoints = []


def play_training_checkpoint(update_count):
    """Capture and display the synth state after a completed optimizer update."""
    checkpoint_loss = float(restoration_loss(raw_controls))
    checkpoint_controls = decode_controls(raw_controls).numpy().copy()
    checkpoint_audio = tf.stop_gradient(render(raw_controls)).numpy().copy()
    audio_checkpoints.append({
        "update": update_count,
        "loss": checkpoint_loss,
        "controls": checkpoint_controls,
        "audio": checkpoint_audio,
    })
    control_text = ", ".join(
        f"{name}={value:.2f}" for name, value in zip(CONTROL_NAMES, checkpoint_controls)
    )
    label = "damaged start" if update_count == 0 else f"after update {update_count}"
    print(f"\nListen — {label} | loss={checkpoint_loss:.6f} | {control_text}")
    display(Audio(checkpoint_audio, rate=SAMPLE_RATE, normalize=False))


print("Target reference — compare every checkpoint with this sound:")
display(Audio(target_audio.numpy(), rate=SAMPLE_RATE, normalize=False))
play_training_checkpoint(0)

with tf.device("/CPU:0"):
    for step in range(STEPS):
        with tf.GradientTape() as tape:
            loss = restoration_loss(raw_controls)
        gradient = tape.gradient(loss, raw_controls)
        loss_history.append(float(loss))
        knob_history.append(decode_controls(raw_controls).numpy().copy())
        optimizer.apply_gradients([(gradient, raw_controls)])

        completed_updates = step + 1
        if completed_updates % LISTEN_EVERY == 0 or completed_updates == STEPS:
            play_training_checkpoint(completed_updates)

loss_history.append(float(restoration_loss(raw_controls)))
knob_history.append(decode_controls(raw_controls).numpy().copy())
# 221 scalar losses -> (221,); 221 four-knob snapshots -> (221, 4).
# Keep update and control axes separate so each plot can follow one knob through time.
loss_history = np.asarray(loss_history)
knob_history = np.asarray(knob_history)
restored_controls = decode_controls(raw_controls).numpy()
restored_audio = tf.stop_gradient(render(raw_controls))

expected_checkpoints = [0, *range(LISTEN_EVERY, STEPS, LISTEN_EVERY), STEPS]
assert [checkpoint["update"] for checkpoint in audio_checkpoints] == expected_checkpoints
print(f"\nCompleted {STEPS} updates on /CPU:0")
print(f"Played {len(audio_checkpoints)} learning checkpoints: {expected_checkpoints}")
print(f"Loss: {loss_history[0]:.6f} -> {loss_history[-1]:.8f}")
for name, start, restored, target in zip(CONTROL_NAMES, damaged_controls.numpy(), restored_controls, target_controls.numpy()):
    print(f"  {name:7s}: {start:.3f} -> {restored:.3f} (target {target:.3f})")

### Did it recover the sound, the controls, or just one score?

A lower combined loss can hide different compromises. The next cell checks four views instead of declaring victory from a downward curve:

- **Objective:** final loss must be below 2% of initial loss.
- **Waveform correlation:** target and restored samples must correlate above `0.995`. Correlation checks matching shape, but can stay high despite a gain mismatch.
- **Spectrum distance:** mean squared log-spectrum error must be below 5% of the damaged version's error.
- **Knob MAE:** the average absolute difference from the four known target settings must be below `0.08`. These raw control-unit differences are a teaching summary, not a perceptual distance.

Those numbers are **assertion thresholds**, not results claimed in advance. Read the actual printed values from your run. A failing assertion is evidence to investigate, not a reason to rewrite a success story.

> **Visual to add — A matching sound need not identify the preset**
>
> Show the fixed target and candidate phrase aligned on the same sample ruler, with a separate four-dial comparison against [1.30,0.72,1.18,1.55]. Contrast waveform agreement, spectrum agreement, and control agreement as separate measurements. Add a small alternate-dial sketch labelled “possible ambiguity, not a measured result.” Do not draw a causal arrow from high correlation to identical controls or unique recovery.
>
> **Proposed caption:** “Matching the output and finding the original settings are different claims.”

Known target controls make that last check possible in this synthetic experiment. A real lost preset may have several settings that sound alike, a different renderer, noise, or misaligned timing. This lab does not test those cases.

In [ ]:
def log_spectrum_distance(audio_a, audio_b):
    # Each (8960,) waveform -> (4481,) spectrum: compare corresponding bins,
    # then average to one error instead of mixing waveform and frequency axes.
    spectrum_a = np.log1p(20.0 * np.abs(np.fft.rfft(audio_a)))
    spectrum_b = np.log1p(20.0 * np.abs(np.fft.rfft(audio_b)))
    return float(np.mean((spectrum_a - spectrum_b) ** 2))

target_np = target_audio.numpy()
damaged_np = damaged_audio.numpy()
restored_np = restored_audio.numpy()
# corrcoef treats the two (8960,) signals as rows and returns a (2, 2) correlation table;
# [0, 1] selects target-vs-restored agreement, not either signal's self-correlation.
waveform_correlation = float(np.corrcoef(target_np, restored_np)[0, 1])
damaged_spectrum_error = log_spectrum_distance(damaged_np, target_np)
restored_spectrum_error = log_spectrum_distance(restored_np, target_np)
knob_mae = float(np.mean(np.abs(restored_controls - target_controls.numpy())))
loss_reduction = 1.0 - loss_history[-1] / loss_history[0]
spectrum_reduction = 1.0 - restored_spectrum_error / damaged_spectrum_error

print(f"Loss reduction:       {loss_reduction:.2%}")
print(f"Waveform correlation: {waveform_correlation:.6f}")
print(f"Spectrum reduction:   {spectrum_reduction:.2%}")
print(f"Mean knob error:      {knob_mae:.4f}")
assert loss_history[-1] < 0.02 * loss_history[0], "Loss did not fall by at least 98%."
assert waveform_correlation > 0.995, "Restored waveform correlation is too low."
assert restored_spectrum_error < 0.05 * damaged_spectrum_error, "Spectrum did not recover enough."
assert knob_mae < 0.08, "Recovered controls are not close enough to the hidden preset."
print("Recovery gates passed.")

## 6 · Put the evidence beside what you hear

The four panels answer different questions:

1. **First 420 ms of waveform:** do corresponding sample shapes line up? All three clips use the same time mask, so the plot does not shift a candidate to make it match.
2. **Magnitude spectrum:** did frequency balance recover? The plotted dB conversion is for viewing; the optimized spectrum loss uses `log1p`, not these dB values.
3. **Loss by update:** how did the scalar objective change? A log vertical axis makes multiplicative improvements easier to see; individual Adam steps need not decrease loss monotonically.
4. **Decoded knobs by update:** where did each audible control travel? Dotted lines are configured targets; solid lines are recorded states, not interpolated ideal paths.

Finally, hear the actual restored array. No clean-looking curve replaces that comparison. These plots and players are produced by execution; no numerical recovery result is embedded in this prose.

In [ ]:
figure, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)
# One (8960,) boolean mask selects matching time/audio samples in all three waveforms;
# each plotted slice is (visible_samples,), so the comparison cannot shift their timing.
view = time_axis <= 0.42
axes[0, 0].plot(time_axis[view], target_np[view], label="target", color="#176B87", lw=1.8)
axes[0, 0].plot(time_axis[view], damaged_np[view], label="damaged", color="#C8553D", alpha=0.70)
axes[0, 0].plot(time_axis[view], restored_np[view], label="restored", color="#2E8B57", lw=1.1, ls="--")
axes[0, 0].set(title="Waveforms: first 420 ms", xlabel="Time (s)", ylabel="Amplitude")
axes[0, 0].legend()

# rfftfreq and each rfft below give matching (4481,) entries, so every
# spectrum value is plotted at its own frequency rather than at a sample index.
spectrum_frequency = np.fft.rfftfreq(target_np.size, 1.0 / SAMPLE_RATE)
for audio, label, color in [(target_np, "target", "#176B87"), (damaged_np, "damaged", "#C8553D"), (restored_np, "restored", "#2E8B57")]:
    magnitude_db = 20.0 * np.log10(np.abs(np.fft.rfft(audio)) + 1e-5)
    axes[0, 1].plot(spectrum_frequency, magnitude_db, label=label, color=color, alpha=0.88)
axes[0, 1].set(xlim=(60, 2_600), ylim=(-65, 35), title="Magnitude spectra", xlabel="Frequency (Hz)", ylabel="Magnitude (dB)")
axes[0, 1].legend()

axes[1, 0].semilogy(loss_history, color="#4C3A69", lw=2)
axes[1, 0].set(title="Restoration objective", xlabel="Optimizer update", ylabel="Loss (log scale)")

knob_colors = ["#176B87", "#D17B0F", "#2E8B57", "#8D5A97"]
for index, (name, color) in enumerate(zip(CONTROL_NAMES, knob_colors)):
    # (221, 4) -> (221,): follow one control across updates, with its matching scalar target.
    axes[1, 1].plot(knob_history[:, index], label=name, color=color, lw=1.8)
    axes[1, 1].axhline(target_controls.numpy()[index], color=color, ls=":", alpha=0.75)
axes[1, 1].set(title="Decoded knob trajectories", xlabel="Optimizer update", ylabel="Knob value")
axes[1, 1].legend(ncol=2)
# (2, 2) subplot grid -> four axes, so the same grid styling reaches every panel.
for axis in axes.flat:
    axis.grid(alpha=0.22)
plt.show()

print("Restored result:")
display(Audio(restored_np, rate=SAMPLE_RATE, normalize=False))

## The short version

- Fixed notes can sound different because of the renderer's settings, not the sequence.
- FFT changes representation so tone controls can act on frequency components; inverse FFT returns to samples.
- Smooth bounded controls keep settings valid, but gradients are taken with respect to their **raw** variables.
- The loss compares audio, not secret knob answers. Backpropagation reports sensitivity; Adam changes parameters.
- Two-sided nudges check all four local derivatives. Listening and separate waveform, spectrum, and control checks judge the outcome.
- One synthetic phrase and a known renderer make a clean inverse problem. They do not demonstrate general audio restoration or unique preset recovery.

**Where we stopped**

- **Built and checked by the cells:** a fixed harmonic phrase, differentiable EQ/drive, four autodiff slopes with finite-difference checks, 220 Adam updates, recorded listening states, and explicit recovery assertions.
- **Talked through:** frequency masks, raw-to-bounded parameterization, loss weighting, correlation's limits, and identifiability.
- **Left out:** perceptual hearing models, unknown note recovery, multi-phrase evaluation, and a real plugin's effects chain.

## What if the notes themselves are missing?

This lab learned **how known notes should sound**. It never chose the next note. The [sequence-memory notebook](../../../03-language-model-foundations/01-sequence-memory/01A-sequence-memory-theory.ipynb) asks the complementary question: given earlier symbols, **what comes next?**

The later [Cinematic Piano Memory capstone](../../../03-language-model-foundations/01-sequence-memory/01C-cinematic-piano-memory.ipynb) makes a sequence model's predicted notes audible. The connection is straightforward:

```text
previous note tokens -> sequence model -> next note/duration
                    -> event list -> renderer -> sound
```

The renderer handles timbre; the sequence model handles order. Connecting them for playback does not automatically make the whole pipeline differentiable: a discrete token choice such as argmax breaks the ordinary gradient path.

For the required route, continue to [The Deep-Learning Map](../../../02-pytorch-foundations/01-deep-learning-map/01-deep-learning-map.ipynb). You have now seen the same error-to-gradient-to-update loop in both a small neural calculation and an audible optimization.